# Module 02 — Neurons, Layers, MLPs

This notebook is about exactly one thing: **how a neural network is structured**. Module 01 built the autograd engine (the `Value` class); this notebook uses it to compose the shapes we call a "neural network" — but does **not** train anything yet. No loss function, no gradient descent, just forward passes, so the only thing in focus is: what is a neuron, a layer, and an MLP, and how do they compose?

Training comes in Module 03.

In [ ]:
# Same Value class as Module 01 (autograd_engine.ipynb) — copied in so this
# notebook runs standalone. Not re-explained here; see Module 01 for that.
import math
import random


class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f"**{power}")

        def _backward():
            self.grad += (power * self.data ** (power - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## A single neuron

A neuron takes several numbers in, and produces one number out:
1. Multiply each input by its own **weight** (one weight per input — this controls how much that input matters).
2. Sum those products, and add a **bias** (a learnable offset).
3. Pass the result through a **nonlinearity** (`tanh` here), which squashes it into a bounded range and — critically — is what lets stacks of neurons represent more than just a straight line/plane.

The weights and bias are the neuron's **parameters** — the numbers that get adjusted during training (Module 03). Right now we just initialize them randomly and look at what the neuron computes.

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

In [ ]:
random.seed(42)

n = Neuron(nin=3)
x = [Value(1.0), Value(-2.0), Value(0.5)]
out = n(x)

print("weights:", n.w)
print("bias:   ", n.b)
print("output: ", out)
print("parameter count:", len(n.parameters()))

## A layer

A layer is just several neurons, all looking at the *same* input, each producing its own output. A layer with 3 inputs and 5 neurons turns a 3-number input into a 5-number output — each of the 5 numbers computed by a differently-weighted neuron.

In [ ]:
class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [ ]:
layer = Layer(nin=3, nout=5)
out = layer(x)

print(f"input has {len(x)} numbers, output has {len(out)} numbers")
print("output:", out)
print("parameter count:", len(layer.parameters()), "(= 5 neurons * (3 weights + 1 bias))")

## An MLP (multi-layer perceptron)

An MLP is a stack of layers, where each layer's output becomes the next layer's input. `MLP(3, [4, 4, 1])` means: start with 3 inputs, go through a layer of 4 neurons, then another layer of 4 neurons, then a final layer of 1 neuron — that last number is the network's overall output.

One detail: the **last** layer usually has `nonlin=False`. Every hidden layer squashes its output through `tanh`, but the final layer is left as a raw number (often called a *logit* or *score*) — squashing it too would arbitrarily cap what the network can output, which matters once we attach a loss function to that output in Module 03.

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [
            Layer(sizes[i], sizes[i + 1], nonlin=(i != len(nouts) - 1))
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
model = MLP(nin=2, nouts=[8, 8, 1])

sample_input = [Value(0.5), Value(-0.3)]
out = model(sample_input)

print("output:", out)
print("total parameter count:", len(model.parameters()))
for i, layer in enumerate(model.layers):
    print(f"  layer {i}: {len(layer.neurons)} neurons, {len(layer.parameters())} parameters, nonlin={layer.neurons[0].nonlin}")

## What just happened

We composed `Value`s into a `Neuron` (weighted sum + bias + nonlinearity), several `Neuron`s into a `Layer` (same input, many outputs), and several `Layer`s into an `MLP` (each layer's output feeds the next). Every number involved — every weight, every bias — is a `Value`, which means the *entire network*, no matter how deep, is one big computation graph built from the primitives in Module 01.

We haven't trained anything: the weights above are random, so the network's output is meaningless right now. That's the point — this notebook was only about structure.

**Next: Module 03** — attach a loss function to this network's output and use `.backward()` + gradient descent to actually teach it something.